In [ ]:
#| default_exp entities

In [ ]:
#| export
import re, json
from functools import lru_cache
from fastcore.all import first

@lru_cache(1<<16)
def _toks(s): return frozenset(re.findall(r'[A-Za-z0-9]+', (s or '').lower()))
@lru_cache(1<<16)
def _acr(s):  return ''.join(w[0] for w in (s or '').split() if w)
def _nums(s): return frozenset(re.findall(r'\d+', s or ''))
def _lex_ok(a, b, lex=0.34, cover=0.5):
    "Guard a proposed merge: token/acronym overlap AND identical numbers, so 'python 3.11' never eats '3.12'."
    if a == b: return True
    A, B = _toks(a), _toks(b); ok = False
    if A and B:
        inter = len(A & B)
        ok = ((inter == min(len(A), len(B)) and inter/max(len(A), len(B)) >= cover) or inter/(len(A)+len(B)-inter) >= lex)
    if not ok and not (_acr(a) == b.lower() or _acr(b) == a.lower()): return False
    return _nums(a) == _nums(b)

def norm_ent(name, canon=None):
    "Canonical entity name: `canon(name)` when it matches, else lowercased and whitespace-collapsed."
    return (canon(name) if canon else None) or ' '.join((name or '').lower().split())

GEN_SP = """Extract a typed knowledge graph from one chunk of any document (paper, book, scripture, playbook, contract).
entities: {name, type}, type whatever fits (concept, method, person, work, model, deity, verse, framework, ...).
relations: {src, rel, dst}, rel a short verb: cites, refers_to, builds_on, defines, part_of, uses, contrasts_with.
Capture EVERY cross-reference to another work, section, verse, citation marker, person or named idea.
Output STRICT JSON only: {"summary": str, "entities":[{"name":str,"type":str}], "relations":[{"src":str,"rel":str,"dst":str}]}"""

_REL_MAP = {'cross_ref':'refers_to','reference':'refers_to','refer_to':'refers_to','cite':'cites','citing':'cites',
            'build_on':'builds_on','uses_':'uses','part-of':'part_of','defined_by':'defines'}
def _norm_rel(r):
    r = (r or '').lower().strip().replace(' ', '_'); return _REL_MAP.get(r, r) if r else None
def _chat_text(r): return r.get('content','') if isinstance(r, dict) else (r if isinstance(r, str) else str(r))
def _parse_json(txt):
    "First JSON object in a model reply, tolerant of fences and trailing prose."
    m = re.search(r'\{.*\}', txt or '', re.S)
    for a in ([m.group(0), m.group(0).replace('\n',' ')] if m else []):
        try: return json.loads(a)
        except Exception: pass
    return {}

def extract_typed(chunks, chat, hints=None):
    "Per chunk {id, summary, entities, relations} via `chat.oneshot(user, sp=GEN_SP)`. `hints[i]` primes reuse of existing names."
    out = []
    for i, c in enumerate(chunks):
        user = f"CHUNK:\n# {c.get('heading') or ''}\n{(c['content'] or '')[:1400]}"
        h = (hints or [None]*len(chunks))[i]
        if h: user = f"Known entities you may reuse (use the EXACT name if it matches, else create new): {h}\n\n" + user
        j = _parse_json(_chat_text(chat.oneshot(user, sp=GEN_SP, max_tokens=640)))
        ents = [e for e in j.get('entities',[]) if isinstance(e,dict) and e.get('name')]
        rels = [dict(r, rel=nr) for r in j.get('relations',[])
                if isinstance(r,dict) and r.get('src') and r.get('dst') and (nr:=_norm_rel(r.get('rel')))]
        out.append(dict(id=c['id'], summary=j.get('summary',''), entities=ents, relations=rels))
    return out

CITE = [
    (re.compile(r'\bArticle\s+(\d+)', re.I), lambda m: f'Article {m.group(1)}'),
    (re.compile(r'\bAnnex(?:es)?\s+([IVXLCDM]+|\d+)', re.I), lambda m: f'Annex {m.group(1).upper()}'),
    (re.compile(r'\bChapter\s+([IVXLCDM]+|\d+)', re.I), lambda m: f'Chapter {m.group(1).upper()}'),
    (re.compile(r'\bTitle\s+([IVXLCDM]+|\d+)', re.I), lambda m: f'Title {m.group(1).upper()}'),
    (re.compile(r'\b(Regulation|Directive)s?\s*\((?:EU|EC|EEC)\)\s*(?:No\s*)?(\d+/\d+)', re.I), lambda m: f'{m.group(1).title()} {m.group(2)}'),
]
def norm_cite(name):
    "Canonical form of a structured legal citation, or None. Pass as `canon=` for a legal corpus."
    for pat, f in CITE:
        if (m := pat.search(str(name))): return f(m)
    return None
def cites(text):
    "Every structured citation in a span, deduped. Pass as `seed_fn=` to seed refers_to edges for a legal corpus."
    return list(dict.fromkeys(f(m) for pat, f in CITE for m in pat.finditer(text or '')))

In [ ]:
#| hide
assert _lex_ok('multi-head attention','multi head attention') and not _lex_ok('python 3.11','python 3.12')
assert norm_ent('The  Transformer') == 'the transformer'
assert norm_ent('Article 3(1)', norm_cite) == 'Article 3'
assert cites('per Article 5 and Annex III') == ['Article 5','Annex III']
class _C:
    def oneshot(self, u, **k): return '{"summary":"s","entities":[{"name":"X","type":"concept"}],"relations":[{"src":"X","rel":"cross_ref","dst":"Y"}]}'
r = extract_typed([{'id':'c1','heading':'h','content':'body'}], _C())[0]
assert r['id']=='c1' and r['relations'][0]['rel']=='refers_to'   # synonym normalised